# RUNECLAW V3 — Opus-Enhanced Training Notebook

Train the RUNECLAW trading AI on the **Opus 4.8 enriched dataset** (9,774+ samples).

**What's new in V3:**
- Optimized for `combined_training_opus.jsonl` (Opus 4.8 knowledge distillation data)
- 10 training categories: trade analysis, JSON bot output, order flow, backtest interpretation, adversarial rejections
- Updated system prompt (22 risk checks, correct indicator weights)
- Fixed Claude model IDs (`claude-opus-4-8`, `claude-sonnet-4-6`, `claude-haiku-4-5`)
- Better test prompts covering all Opus categories
- Dual GGUF export: Q4_K_M (fast, 5GB) + Q8_0 (quality, 8GB — fits RTX 4080)
- Model name: `runeclaw-v3`

**Setup:** Runtime → Change runtime type → **L4 GPU** (or A100 if available, use credits)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install safetensors sentencepiece protobuf
print("Dependencies installed.")

## 2. GPU Detection & Hyperparameter Selection
Auto-selects the best configuration for your GPU. L4 gets rank-64, A100 gets rank-128.

In [ ]:
import torch
import os
import json
import random
import time

random.seed(42)

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1024**3

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# ── Auto-select best config for available GPU ──
if vram_gb >= 70:
    # A100 80GB — max everything
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 128
    LORA_ALPHA = 128
    BATCH_SIZE = 8
    GRAD_ACCUM = 2
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 4e-5
    tier = "A100-80GB MAX"
elif vram_gb >= 35:
    # A100 40GB
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 128
    LORA_ALPHA = 128
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 4e-5
    tier = "A100-40GB"
elif vram_gb >= 20:
    # L4 / A10
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 64
    LORA_ALPHA = 64
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    MAX_SEQ = 2048
    NUM_EPOCHS = 3
    LEARNING_RATE = 5e-5
    tier = "L4/A10"
elif vram_gb >= 14:
    # T4
    MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    LORA_RANK = 32
    LORA_ALPHA = 32
    BATCH_SIZE = 2
    GRAD_ACCUM = 8
    MAX_SEQ = 1024
    NUM_EPOCHS = 3
    LEARNING_RATE = 5e-5
    tier = "T4"
else:
    MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    LORA_RANK = 16
    LORA_ALPHA = 16
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    MAX_SEQ = 512
    NUM_EPOCHS = 3
    LEARNING_RATE = 5e-5
    tier = "Low VRAM"

WARMUP_RATIO = 0.05
MODEL_NAME = "runeclaw-v3"

# Updated system prompt — matches bot config (22 risk checks, correct weights)
SYSTEM_PROMPT = (
    "You are RUNECLAW, an AI trading analyst. You analyze cryptocurrency "
    "markets using the GetClaw Confluence Engine (12 weighted indicators: "
    "RSI-14 w=1.5, MACD w=1.0, Bollinger w=1.2, EMA Cross w=1.0, Volume "
    "Profile w=1.3, OBV w=0.8, Stoch RSI w=0.7, ADX w=1.1, Ichimoku w=0.9, "
    "VWAP w=1.4, ATR w=0.8, Fibonacci w=1.0), enforce strict risk management "
    "through 22 automated checks (all must pass, fail-closed), and generate "
    "structured trade ideas. You never execute without human confirmation. "
    "Capital preservation above all."
)

eff_batch = BATCH_SIZE * GRAD_ACCUM
print(f"\n{'='*50}")
print(f"  Config: {tier}")
print(f"{'='*50}")
print(f"  Model:     {MODEL}")
print(f"  LoRA:      rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"  Context:   {MAX_SEQ} tokens")
print(f"  Batch:     {BATCH_SIZE} x {GRAD_ACCUM} = {eff_batch} effective")
print(f"  Epochs:    {NUM_EPOCHS}")
print(f"  LR:        {LEARNING_RATE}")
print(f"  Warmup:    {WARMUP_RATIO*100:.0f}% of steps")

## 3. Load Training Data

**Upload your `combined_training_all.jsonl`** (~50K+ samples: full Claude dataset + 1,012 Opus 4.8)

Created by running `python merge_opus_claude.py` locally. If you don't have the file, synthetic data will be generated as fallback.

In [ ]:
# ── Upload your merged training data ──
from google.colab import files

DATA_FILE = "/content/training_data.jsonl"
all_samples = []

print("Upload your combined_training_all.jsonl (~50K samples):")
print("(This is combined_training_claude.jsonl + opus_training.jsonl merged)")
print("(Click Cancel to generate synthetic data instead)\n")
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        lines = content.decode('utf-8').strip().split('\n')
        for line in lines:
            line = line.strip()
            if line:
                try:
                    sample = json.loads(line)
                    if 'instruction' in sample and 'output' in sample:
                        all_samples.append(sample)
                except json.JSONDecodeError:
                    pass
        print(f"Loaded {len(all_samples):,} samples from {fname}")
except Exception as e:
    print(f"No file uploaded. Will generate synthetic data in next cell.")

if all_samples:
    print(f"\nDataset ready: {len(all_samples):,} samples")
    # Quality stats
    approved = sum(1 for s in all_samples if 'APPROVED' in s.get('output','').upper())
    rejected = sum(1 for s in all_samples if 'REJECTED' in s.get('output','').upper())
    json_fmt = sum(1 for s in all_samples if s.get('output','').strip().startswith('{'))
    avg_len = sum(len(s.get('output','')) for s in all_samples) / len(all_samples)

    print(f"  Approved:    {approved:,}")
    print(f"  Rejected:    {rejected:,}")
    print(f"  JSON format: {json_fmt:,}")
    print(f"  Avg output:  {avg_len:.0f} chars")
    GENERATE_SYNTHETIC = False
else:
    GENERATE_SYNTHETIC = True
    print("Will generate synthetic data in next cell.")

In [ ]:
# ── Synthetic data generation (only runs if no file was uploaded) ──
if GENERATE_SYNTHETIC:
    print("Generating synthetic training data...")

    PAIRS = [
        "BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT", "XRP/USDT",
        "ADA/USDT", "AVAX/USDT", "DOGE/USDT", "DOT/USDT", "MATIC/USDT",
        "LINK/USDT", "UNI/USDT", "ATOM/USDT", "FIL/USDT", "APT/USDT",
        "ARB/USDT", "OP/USDT", "SUI/USDT", "SEI/USDT", "INJ/USDT",
        "TIA/USDT", "JUP/USDT", "WIF/USDT", "NEAR/USDT", "RENDER/USDT",
        "FET/USDT", "RUNE/USDT", "STX/USDT", "ICP/USDT", "PEPE/USDT",
    ]

    INDICATORS = [
        "RSI", "MACD", "Bollinger Bands", "EMA Cross", "Volume Profile",
        "OBV", "Stochastic RSI", "ADX", "Ichimoku Cloud", "VWAP",
        "Fibonacci Retracement", "ATR",
    ]

    RISK_CHECKS = [
        "max_position_size", "portfolio_heat", "correlation_limit",
        "drawdown_threshold", "volatility_filter", "liquidity_check",
        "spread_limit", "funding_rate", "open_interest_divergence",
        "whale_activity", "exchange_reserve", "news_sentiment",
        "time_of_day", "consecutive_losses", "win_rate_threshold",
        "max_leverage", "stop_loss_required", "take_profit_required",
        "risk_reward_minimum", "daily_loss_limit", "weekly_loss_limit",
        "margin_utilization", "circuit_breaker",
    ]

    REGIMES = ["TRENDING_UP", "TRENDING_DOWN", "RANGING", "VOLATILE",
               "CHOPPY", "ACCUMULATION", "DISTRIBUTION", "STRONG_TREND"]
    REGIME_MULTIPLIERS = {
        "CHOPPY": 0.5, "RANGING": 0.7, "VOLATILE": 0.3,
        "TRENDING_UP": 1.0, "TRENDING_DOWN": 1.0,
        "STRONG_TREND": 1.5, "ACCUMULATION": 0.8, "DISTRIBUTION": 0.6,
    }

    PRICES = {
        "BTC": 68000, "ETH": 3800, "SOL": 180, "BNB": 620, "XRP": 0.62,
        "ADA": 0.48, "AVAX": 38, "DOGE": 0.16, "DOT": 7.5, "MATIC": 0.72,
        "LINK": 18, "UNI": 12, "ATOM": 9.5, "FIL": 6.2, "APT": 9.8,
        "ARB": 1.2, "OP": 2.8, "SUI": 1.4, "SEI": 0.45, "INJ": 28,
        "TIA": 8.5, "JUP": 1.1, "WIF": 2.8, "PEPE": 0.000012, "RENDER": 8.5,
        "FET": 2.3, "NEAR": 7.2, "ICP": 14, "RUNE": 5.5, "STX": 2.1,
    }

    def rp(base, pct=0.1):
        raw = base * (1 + random.uniform(-pct, pct))
        if base < 0.01: return round(raw, 8)
        elif base < 1: return round(raw, 6)
        elif base < 100: return round(raw, 4)
        return round(raw, 2)

    def rpct(lo, hi): return round(random.uniform(lo, hi), 2)
    def gpp(pair): return PRICES.get(pair.split('/')[0], 10.0)

    # ── 1. Trade Analysis with regime-linked sizing (5000) ──
    for _ in range(5000):
        pair = random.choice(PAIRS)
        tf = random.choice(["15m","1h","4h","1d"])
        d = random.choice(["long", "short"])
        regime = random.choice(REGIMES)
        mult = REGIME_MULTIPLIERS[regime]
        base = gpp(pair); entry = rp(base, 0.02)
        if entry == 0: entry = base
        base_risk = rpct(1.5, 3.0)  # base position %
        final_risk = round(base_risk * mult, 2)  # regime-adjusted
        if d == "long":
            stop = round(entry * (1 - random.uniform(0.01, 0.04)), 6)
            tp1 = round(entry * (1 + random.uniform(0.02, 0.06)), 6)
        else:
            stop = round(entry * (1 + random.uniform(0.01, 0.04)), 6)
            tp1 = round(entry * (1 - random.uniform(0.02, 0.06)), 6)
        risk_pct = round(abs(entry - stop) / entry * 100, 2) if entry > 0 else 1.0
        reward_pct = round(abs(tp1 - entry) / entry * 100, 2) if entry > 0 else 1.0
        rr = round(reward_pct / risk_pct, 2) if risk_pct > 0 else 0
        conf = rpct(0.45, 0.95)
        ni = random.randint(6, 12)
        bl = random.randint(3, ni); brl = ni - bl
        ai = random.sample(INDICATORS, ni)
        out = (f"Trade Idea: {pair}\nDirection: {d.upper()}\nTimeframe: {tf}\n"
               f"Market Regime: {regime}\n\n"
               f"Entry: {entry}\nStop Loss: {stop} ({risk_pct}%)\n"
               f"Take Profit 1: {tp1} ({reward_pct}%)\nRisk:Reward: 1:{rr}\n\n"
               f"Confluence: {conf} ({ni} indicators)\n"
               f"Bullish ({bl}): {', '.join(ai[:bl])}\n"
               f"Bearish ({brl}): {', '.join(ai[bl:])}\n\n"
               f"Position Size: {final_risk}% of portfolio\n"
               f"  Base risk: {base_risk}% x regime multiplier {mult}x ({regime}) = {final_risk}%\n"
               f"Status: {'APPROVED' if conf > 0.6 and rr >= 1.5 else 'REJECTED'}")
        all_samples.append({"instruction": f"Analyze {pair} on {tf} for trade setups.", "input": "", "output": out})

    # ── 2. JSON format trades for bot integration (3000) ──
    for _ in range(3000):
        pair = random.choice(PAIRS)
        tf = random.choice(["1h","4h","1d"])
        d = random.choice(["LONG", "SHORT"])
        regime = random.choice(REGIMES)
        base = gpp(pair); entry = rp(base, 0.02)
        if entry == 0: entry = base
        if d == "LONG":
            stop = round(entry * (1 - random.uniform(0.01, 0.04)), 6)
            tp = round(entry * (1 + random.uniform(0.02, 0.06)), 6)
        else:
            stop = round(entry * (1 + random.uniform(0.01, 0.04)), 6)
            tp = round(entry * (1 - random.uniform(0.02, 0.06)), 6)
        conf = rpct(0.50, 0.95)
        rr_val = round(abs(tp - entry) / abs(entry - stop), 2) if abs(entry - stop) > 0 else 0
        signals = random.sample(INDICATORS, random.randint(5, 10))
        reasoning = random.choice([
            f"{pair} showing strong {d.lower()} momentum with ADX > 25 and MACD crossover.",
            f"Volume surge with OBV confirmation on {tf}. {regime} regime favors {d.lower()} entries.",
            f"EMA 20/50 aligned {d.lower()}, RSI at {random.randint(30,70)}, BB squeeze breakout.",
            f"Multi-timeframe alignment: 1H/4H/1D all {d.lower()}. High confluence setup.",
            f"Fibonacci {random.choice(['0.618','0.5','0.382'])} retracement holding as support. {d} bias confirmed.",
        ])
        json_out = json.dumps({
            "direction": d,
            "confidence": conf,
            "entry_price": entry,
            "stop_loss": stop,
            "take_profit": tp,
            "risk_reward": rr_val,
            "regime": regime,
            "signals_used": signals,
            "reasoning": reasoning,
            "verdict": "APPROVED" if conf > 0.6 and rr_val >= 1.5 else "REJECTED"
        }, indent=2)
        all_samples.append({"instruction": f"Analyze {pair} on {tf}. Return JSON.", "input": "", "output": json_out})

    # ── 3. No-Trade Rejections (4000) ──
    reject_reasons = [
        "Low confluence ({}/12 indicators)",
        "R:R {:.1f}:1 below 1.5:1 minimum",
        "Portfolio heat {}% exceeds 6% max",
        "Consecutive losses: {} (max 3)",
        "Daily drawdown {}% exceeds 2% limit",
        "Spread {}% exceeds 0.1% max",
        "Funding rate {}% — crowded trade",
        "Volume ${}M below $10M threshold",
        "Circuit breaker active",
        "Conflicting timeframe signals",
    ]
    for _ in range(4000):
        pair = random.choice(PAIRS); d = random.choice(["long","short"])
        reason = random.choice(reject_reasons)
        try:
            reason = reason.format(random.randint(2,5), rpct(0.5,1.4), rpct(6.1,12),
                random.randint(3,7), rpct(2.1,5), rpct(0.11,0.5), rpct(0.05,0.3),
                rpct(0.5,9.9))
        except: pass
        nf = random.randint(1, 5)
        fc = random.sample(RISK_CHECKS, nf)
        out = (f"Trade Scan: {pair} ({d.upper()})\nStatus: REJECTED\n\n"
               f"Reason: {reason}\n\nFailed checks ({nf}/{len(RISK_CHECKS)}):\n")
        for c in fc: out += f"  FAIL: {c}\n"
        out += "\nAction: NO TRADE. Capital preservation above all."
        all_samples.append({"instruction": f"Scan {pair} for {d} setups.", "input": "", "output": out})

    # ── 4. Multi-Timeframe Analysis (3000) ──
    for _ in range(3000):
        pair = random.choice(PAIRS)
        tfa = {}
        for tf in ["15m","1h","4h","1d"]:
            tfa[tf] = {"trend": random.choice(["bullish","bearish","neutral"]), "str": rpct(0.2, 0.95)}
        trends = [v["trend"] for v in tfa.values()]
        bc = trends.count("bullish"); brc = trends.count("bearish")
        if bc >= 3: align = "STRONG BULLISH"; d = "LONG"
        elif brc >= 3: align = "STRONG BEARISH"; d = "SHORT"
        elif bc == 2: align = "WEAK BULLISH"; d = "LONG"
        elif brc == 2: align = "WEAK BEARISH"; d = "SHORT"
        else: align = "MIXED"; d = "NONE"
        lines = [f"Multi-Timeframe Analysis: {pair}\n"]
        for tf, data in tfa.items():
            lines.append(f"  {tf}: {data['trend'].upper()} (strength: {data['str']})")
        lines.append(f"\nAlignment: {align} ({max(bc, brc)}/4)")
        if d != "NONE":
            lines.append(f"Direction: {d}")
            lines.append(f"Confirm on 15m for precise entry")
        else:
            lines.append("Direction: NONE — STAND ASIDE, conflicting signals")
        all_samples.append({"instruction": f"Multi-timeframe analysis on {pair}.", "input": "", "output": "\n".join(lines)})

    # ── 5. Confluence Breakdown (3000) ──
    for _ in range(3000):
        pair = random.choice(PAIRS); tf = random.choice(["1h","4h","1d"])
        scores = {}
        for ind in INDICATORS:
            sig = random.choice(["bullish","bearish","neutral"])
            scores[ind] = {"sig": sig, "w": round(random.uniform(0.5, 2.0), 1), "c": rpct(0.3, 0.99)}
        tw = sum(v["w"] for v in scores.values())
        bw = sum(v["w"] for v in scores.values() if v["sig"] == "bullish")
        brw = sum(v["w"] for v in scores.values() if v["sig"] == "bearish")
        conf = round(max(bw, brw) / tw, 2) if tw > 0 else 0
        lines = [f"GetClaw Confluence: {pair} ({tf})\n{'='*45}\n"]
        for ind, d in scores.items():
            icon = "+" if d["sig"] == "bullish" else "-" if d["sig"] == "bearish" else "~"
            lines.append(f"  [{icon}] {ind}: {d['sig'].upper()} (w:{d['w']}, conf:{d['c']})")
        lines.extend([f"\nConfluence Score: {conf}",
            f"Signal: {'BULLISH' if bw > brw else 'BEARISH' if brw > bw else 'NEUTRAL'}",
            f"Actionable: {'YES' if conf > 0.6 else 'NO'}"])
        all_samples.append({"instruction": f"GetClaw confluence for {pair} on {tf}.", "input": "", "output": "\n".join(lines)})

    # ── 6. Risk Edge Cases + Circuit Breakers (2000) ──
    for _ in range(2000):
        pair = random.choice(PAIRS)
        scenario = random.choice(["flash crash","max drawdown","correlation spike",
            "liquidation cascade","exchange downtime","funding spike","stablecoin depeg"])
        out = (f"CIRCUIT BREAKER: {scenario.upper()}\nPair: {pair}\n\n"
               f"Protocol:\n  1. HALT all new orders\n  2. Review open positions\n"
               f"  3. Tighten stops to breakeven\n  4. Reduce exposure by 50%\n"
               f"  5. Await human confirmation\n\n"
               f"Recovery: Wait for ATR normalization, resume at 25% size.\n"
               f"Capital preservation is non-negotiable.")
        all_samples.append({"instruction": f"Handle {scenario} for {pair}.", "input": "", "output": out})

    # ── 7. Position Sizing with Regime Math (3000) ──
    for _ in range(3000):
        pair = random.choice(PAIRS); base = gpp(pair)
        port = random.choice([1000, 5000, 10000, 25000, 50000, 100000])
        regime = random.choice(REGIMES)
        mult = REGIME_MULTIPLIERS[regime]
        base_risk_pct = rpct(1.0, 2.5)
        final_risk_pct = round(base_risk_pct * mult, 2)
        entry = rp(base, 0.02)
        if entry == 0: entry = base
        sp = rpct(1.0, 4.0); stop = round(entry * (1 - sp / 100), 6)
        risk_amount = round(port * final_risk_pct / 100, 2)
        price_risk = abs(entry - stop)
        units = round(risk_amount / price_risk, 4) if price_risk > 0 else 0
        pos_value = round(units * entry, 2)
        pos_pct = round(pos_value / port * 100, 2) if port > 0 else 0
        out = (f"Position Sizing: {pair}\n{'='*40}\n\n"
               f"Portfolio: ${port:,}\nRegime: {regime}\n\n"
               f"Base risk: {base_risk_pct}% per trade\n"
               f"Regime multiplier: {mult}x ({regime})\n"
               f"Adjusted risk: {base_risk_pct}% x {mult} = {final_risk_pct}%\n"
               f"Risk amount: ${risk_amount:,.2f}\n\n"
               f"Entry: {entry}\nStop Loss: {stop} ({sp}%)\n"
               f"Units: {units}\nPosition value: ${pos_value:,}\n"
               f"Portfolio %: {pos_pct}%\n\n"
               f"{'APPROVED' if pos_pct <= 5 else 'REDUCED to 5% cap'}")
        all_samples.append({"instruction": f"Position size for {pair} with ${port:,} portfolio in {regime} regime.", "input": "", "output": out})

    # ── 8. Market Regime Classification (2000) ──
    regime_data = {
        "TRENDING_UP": ("Buy dips to EMA support", "ADX>25, above EMA 20/50", 1.0),
        "TRENDING_DOWN": ("Short rallies or stand aside", "ADX>25, below EMA 20/50", 1.0),
        "RANGING": ("Mean reversion: buy support, sell resistance", "ADX<20, BB squeeze", 0.7),
        "VOLATILE": ("Reduce exposure, widen stops", "ATR spike, BB expansion", 0.3),
        "CHOPPY": ("Reduce size 50%, A+ setups only", "ADX<15, whipsaw price action", 0.5),
        "ACCUMULATION": ("Scale into longs on volume", "OBV rising, price flat", 0.8),
        "DISTRIBUTION": ("Reduce longs, prepare shorts", "OBV declining, price flat", 0.6),
        "STRONG_TREND": ("Trend-follow with larger size", "ADX>40, all TFs aligned", 1.5),
    }
    for _ in range(2000):
        pair = random.choice(PAIRS)
        regime = random.choice(REGIMES)
        strat, inds, mult = regime_data[regime]
        adx = rpct(10, 60)
        out = (f"Market Regime: {pair}\n{'='*40}\n\n"
               f"Regime: {regime}\n"
               f"ADX: {adx} ({'trending' if adx > 25 else 'ranging'})\n"
               f"Indicators: {inds}\n\nStrategy: {strat}\n"
               f"Position multiplier: {mult}x\n"
               f"Persistence: {'HIGH' if adx > 35 else 'MODERATE' if adx > 20 else 'LOW'}")
        all_samples.append({"instruction": f"Classify market regime for {pair}.", "input": "", "output": out})

    # ── 9. Drawdown Recovery (2000) ──
    for _ in range(2000):
        dd = rpct(1.5, 15); nl = random.randint(2, 10)
        port = random.choice([10000, 25000, 50000, 100000])
        da = round(port * dd / 100, 2)
        if dd < 2: sev = "NORMAL"; act = "Continue with standard parameters."
        elif dd < 5: sev = "ELEVATED"; act = "Reduce size 50%, raise confluence to 0.70, A+ setups only."
        elif dd < 10: sev = "HIGH"; act = "HALT trading. Paper trade 20 signals. Full audit required."
        else: sev = "CRITICAL"; act = "EMERGENCY HALT. Close all. 1 week cooling off. Resume at 10% size."
        out = (f"DRAWDOWN PROTOCOL\n{'='*40}\n\n"
               f"Drawdown: {dd}% (${da:,})\nConsecutive losses: {nl}\n"
               f"Portfolio: ${port - da:,.2f} (from ${port:,})\n\n"
               f"Severity: {sev}\nAction: {act}")
        all_samples.append({"instruction": f"Handle {dd}% drawdown with {nl} consecutive losses.", "input": "", "output": out})

    # ── 10. Chain of Thought (2000) ──
    for _ in range(2000):
        pair = random.choice(PAIRS); d = random.choice(["LONG","SHORT"])
        conf = rpct(0.4, 0.9); rr = rpct(0.8, 4.0)
        heat = rpct(0, 8); cl = random.randint(0, 5)
        regime = random.choice(REGIMES)
        mult = REGIME_MULTIPLIERS[regime]
        ok = conf > 0.6 and rr >= 1.5 and heat < 6 and cl < 3
        out = (f"Chain-of-Thought: {pair} ({d})\n{'='*45}\n\n"
               f"1. Confluence: {conf} {'PASS' if conf > 0.6 else 'FAIL — below 0.6 threshold'}\n"
               f"2. Risk:Reward: 1:{rr} {'PASS' if rr >= 1.5 else 'FAIL — below 1.5:1 minimum'}\n"
               f"3. Portfolio heat: {heat}% {'PASS' if heat < 6 else 'FAIL — exceeds 6% max'}\n"
               f"4. Consecutive losses: {cl} {'PASS' if cl < 3 else 'FAIL — exceeds 3 max'}\n"
               f"5. Regime: {regime} (multiplier: {mult}x)\n\n"
               f"Status: {'APPROVED' if ok else 'REJECTED'}")
        if not ok: out += "\nCapital preservation above all."
        all_samples.append({"instruction": f"Walk through reasoning for {d} on {pair}.", "input": "", "output": out})

    random.shuffle(all_samples)
    print(f"\nGenerated {len(all_samples):,} synthetic samples")
    print(f"  (Includes regime-sizing math + JSON format + rejection examples)")
else:
    print(f"Using uploaded data: {len(all_samples):,} samples (skipped synthetic generation)")

## 4. Claude Enrichment (Optional)

Generate additional expert samples using Claude API. Uses prompt caching for 90% input cost reduction.

In [ ]:
USE_CLAUDE = False  # Set True to generate Claude samples
CLAUDE_SAMPLES = 500
CLAUDE_CONCURRENT = 3

if USE_CLAUDE:
    !pip install -q --upgrade anthropic
    import getpass
    ANTHROPIC_API_KEY = getpass.getpass("Anthropic API key: ")
    print("Key set.")
else:
    print("Claude enrichment disabled. Set USE_CLAUDE = True to enable.")

In [ ]:
if USE_CLAUDE:
    import anthropic
    from concurrent.futures import ThreadPoolExecutor, as_completed

    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    # Updated model IDs — correct naming from console.anthropic.com
    CLAUDE_MODEL = None
    for candidate in ["claude-opus-4-8", "claude-sonnet-4-6", "claude-haiku-4-5",
                       "claude-opus-4-latest", "claude-sonnet-4-latest",
                       "claude-3-5-sonnet-20241022", "claude-3-5-haiku-20241022"]:
        try:
            client.messages.create(model=candidate, max_tokens=10,
                messages=[{"role": "user", "content": "test"}])
            CLAUDE_MODEL = candidate
            print(f"Using: {CLAUDE_MODEL}")
            break
        except Exception:
            continue

    if not CLAUDE_MODEL:
        print("No working Claude model found. Skipping.")
        USE_CLAUDE = False

if USE_CLAUDE:
    _PAIRS = ["BTC/USDT","ETH/USDT","SOL/USDT","BNB/USDT","XRP/USDT",
              "ADA/USDT","AVAX/USDT","DOGE/USDT","LINK/USDT","INJ/USDT"]
    _REGIMES = ["TRENDING_UP","TRENDING_DOWN","RANGING","VOLATILE","CHOPPY","BREAKOUT"]
    _TFS = ["15m","1h","4h","1d"]
    samples_per_cat = CLAUDE_SAMPLES // 7

    prompts = []
    cats = {
        "trade_json": (
            "Analyze {pair} on {tf} and produce a structured JSON trade idea with fields: "
            "direction (LONG/SHORT), confidence (0-1), entry_price, stop_loss, take_profit, "
            "risk_reward, regime, signals_used (list), reasoning, verdict (APPROVED/REJECTED). "
            "Use realistic prices. Apply regime-based position sizing."
        ),
        "trade_text": (
            "Generate a detailed trade analysis for {pair} on {tf}. Include: regime classification, "
            "confluence score from 12 indicators, entry/SL/TP with percentages, R:R ratio, "
            "regime-adjusted position sizing (show the math: base_risk x regime_multiplier = final), "
            "and verdict APPROVED or REJECTED."
        ),
        "rejection": (
            "Analyze {pair} for a {direction} trade on {tf}. The setup SHOULD be REJECTED. "
            "Explain which of the 22 risk checks fail, why confluence is insufficient, "
            "or what market conditions make this dangerous. Be specific with numbers."
        ),
        "regime": (
            "The market for {pair} has transitioned from {regime1} to {regime2}. "
            "Explain: what indicators signaled this, how strategy should adapt, "
            "what position sizing multiplier applies ({regime2} regime), and what adjustments."
        ),
        "risk_mgmt": (
            "A trader wants to {direction} {pair} with ${portfolio:,} portfolio in {regime} regime. "
            "Walk through risk checks. Calculate position size: base risk % x regime multiplier "
            "({regime}) = adjusted risk %. Show dollar amounts and unit counts."
        ),
        "mtf": (
            "Multi-timeframe analysis for {pair} across 15m, 1h, 4h, 1d. For each: trend, strength, "
            "key levels. Assess alignment. Final recommendation: LONG, SHORT, or STAND ASIDE."
        ),
        "educational": (
            "Explain {topic} in crypto trading with the GetClaw Confluence Engine. "
            "Cover: definition, calculation, thresholds, common mistakes, integration."
        ),
    }

    for cat, template in cats.items():
        for i in range(samples_per_cat):
            pair = random.choice(_PAIRS)
            tf = random.choice(_TFS)
            direction = random.choice(["long","short"])
            portfolio = random.choice([5000,10000,25000,50000])
            regime = random.choice(_REGIMES)
            regime1, regime2 = random.sample(_REGIMES, 2)
            topic = random.choice(["confluence scoring","position sizing with Kelly criterion",
                "drawdown recovery","multi-timeframe alignment","circuit breakers",
                "regime detection with ADX","risk:reward optimization","volatility-adjusted stops"])
            prompt = template.format(pair=pair, tf=tf, direction=direction, portfolio=portfolio,
                regime=regime, regime1=regime1, regime2=regime2, topic=topic)
            inst_map = {
                "trade_json": f"Analyze {pair} on {tf}. Return JSON.",
                "trade_text": f"Analyze {pair} on {tf} for trade setups.",
                "rejection": f"Scan {pair} for {direction} setups on {tf}.",
                "regime": f"Market regime changed for {pair}. How to adapt?",
                "risk_mgmt": f"Risk check for {direction} {pair} with ${portfolio:,}.",
                "mtf": f"Multi-timeframe analysis on {pair}.",
                "educational": f"Explain {topic} for crypto trading.",
            }
            prompts.append({"cat": cat, "prompt": prompt, "instruction": inst_map[cat]})

    random.shuffle(prompts)
    print(f"Prepared {len(prompts)} Claude prompts")

    # Use prompt caching for system prompt
    cached_system = [{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}]
    claude_samples = []
    errors = 0

    def call_claude(item):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=1500, temperature=0.7,
                system=cached_system,
                messages=[{"role": "user", "content": item["prompt"]}])
            text = resp.content[0].text.strip()
            if len(text) > 50:
                return {"instruction": item["instruction"], "input": "", "output": text}
        except Exception as e:
            if "rate" in str(e).lower(): time.sleep(5)
        return None

    print(f"Generating with {CLAUDE_CONCURRENT} threads...")
    with ThreadPoolExecutor(max_workers=CLAUDE_CONCURRENT) as executor:
        futures = {executor.submit(call_claude, p): p for p in prompts}
        done = 0
        for future in as_completed(futures):
            done += 1
            result = future.result()
            if result: claude_samples.append(result)
            else: errors += 1
            if done % 25 == 0 or done == len(prompts):
                print(f"  {done}/{len(prompts)} | OK: {len(claude_samples)} | Err: {errors}")

    all_samples.extend(claude_samples)
    random.shuffle(all_samples)
    print(f"\nClaude: {len(claude_samples)} samples. Total: {len(all_samples):,}")
else:
    print(f"Skipped Claude. Dataset: {len(all_samples):,} samples")

In [ ]:
# ── Save final dataset to disk ──
DATA_FILE = "/content/training_data.jsonl"
with open(DATA_FILE, "w", encoding="utf-8") as f:
    for s in all_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

print(f"Saved {len(all_samples):,} samples to {DATA_FILE}")

# Download a copy
from google.colab import files
try:
    files.download(DATA_FILE)
    print("Download started.")
except:
    print("Auto-download not available. Use Files panel to download.")

## 5. Load Model & Apply LoRA

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading {MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Applying LoRA (rank={LORA_RANK})...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

## 6. Tokenize Dataset

In [ ]:
# ── Format all samples into chat template ──
print(f"Formatting {len(all_samples):,} samples...")

formatted_texts = []
for ex in all_samples:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    user_msg = ex["instruction"]
    if ex.get("input", "").strip():
        user_msg += "\n\n" + ex["input"]
    msgs.append({"role": "user", "content": user_msg})
    msgs.append({"role": "assistant", "content": ex["output"]})
    formatted_texts.append(
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    )

# Spot check token lengths
check_n = min(500, len(formatted_texts))
lens = [len(tokenizer(t, truncation=False)["input_ids"]) for t in formatted_texts[:check_n]]
print(f"Token lengths (sample of {check_n}):")
print(f"  Avg: {sum(lens)/len(lens):.0f}, Max: {max(lens)}, Min: {min(lens)}")
print(f"  Under {MAX_SEQ}: {sum(1 for l in lens if l <= MAX_SEQ)}/{check_n}")

In [ ]:
# ── Tokenize into training dataset ──
from torch.utils.data import Dataset as TorchDataset
from torch.nn.utils.rnn import pad_sequence

class TrainDataset(TorchDataset):
    def __init__(self, texts, tok, max_len):
        self.items = []
        print(f"  Tokenizing {len(texts):,} samples...")
        for i, t in enumerate(texts):
            enc = tok(t, truncation=True, max_length=max_len, padding=False, return_tensors="pt")
            ids = enc["input_ids"].squeeze()
            self.items.append({
                "input_ids": ids,
                "attention_mask": torch.ones_like(ids),
                "labels": ids.clone(),
            })
            if (i + 1) % 10000 == 0:
                print(f"    {i+1:,}/{len(texts):,}...")
        print(f"  Done: {len(self.items):,} samples tokenized.")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

class PadCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, batch):
        return {
            "input_ids": pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=self.pad_id),
            "labels": pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100),
            "attention_mask": pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0),
        }

train_dataset = TrainDataset(formatted_texts, tokenizer, MAX_SEQ)
collator = PadCollator(tokenizer.pad_token_id)
print(f"\nDataset ready: {len(train_dataset):,} samples")

## 7. Train

In [ ]:
from transformers import Trainer, TrainingArguments

total_steps = len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f"{'='*50}")
print(f"  TRAINING CONFIG")
print(f"{'='*50}")
print(f"  Samples:     {len(train_dataset):,}")
print(f"  Epochs:      {NUM_EPOCHS}")
print(f"  Total steps: {total_steps:,}")
print(f"  Warmup:      {warmup_steps} steps")
print(f"  Batch:       {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LoRA rank:   {LORA_RANK}")
print(f"  LR:          {LEARNING_RATE}")
print()

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=warmup_steps,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=f"/content/{MODEL_NAME}-checkpoints",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

print("Starting training...")
stats = trainer.train()

print(f"\n{'='*50}")
print(f"  TRAINING COMPLETE")
print(f"{'='*50}")
print(f"  Final loss:  {stats.training_loss:.4f}")
print(f"  Runtime:     {stats.metrics['train_runtime']:.0f}s ({stats.metrics['train_runtime']/3600:.1f}h)")
print(f"  Samples/sec: {stats.metrics.get('train_samples_per_second', 0):.1f}")

## 8. Quick Test
Verify the model produces sensible output before exporting.

In [ ]:
FastLanguageModel.for_inference(model)

# Test prompts covering key Opus training categories
test_prompts = [
    "Analyze BTC/USDT on 4h for trade setups.",
    "Analyze ETH/USDT at $3800 on 1h and output a structured JSON trade decision.",
    "Should RUNECLAW take this SOL/USDT setup? Confluence is 0.52, just below threshold.",
    "Analyze order flow data for BTC/USDT and assess smart money positioning.",
    "Calculate position size for SOL/USDT with $25,000 portfolio in VOLATILE regime.",
    "Interpret backtest: 80 trades, 42% win rate, +0.12R expectancy, 8.5% max drawdown.",
]

for prompt_text in test_prompts:
    test_prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ], tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.3, top_p=0.9,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt_text}")
    print(f"{'='*60}")
    print(response[:800])
    print()

## 9. Export to GGUF
Export Q4_K_M (fast, ~5GB) and Q8_0 (higher quality, ~8GB — fits RTX 4080 16GB).

In [ ]:
OUTPUT_DIR = f"/content/{MODEL_NAME}-gguf"

# ── Export Q4_K_M (fast inference, smaller file) ──
print("Exporting Q4_K_M quantization...")
model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

# List what was actually created
print(f"\nQ4_K_M output directory:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    src = f"{OUTPUT_DIR}/{fn}"
    if os.path.isdir(src):
        print(f"  [DIR] {fn}/")
    else:
        size = os.path.getsize(src)
        if size > 1024**2:
            print(f"  {fn}: {size/1024**3:.2f} GB")
        else:
            print(f"  {fn}: {size/1024:.1f} KB")

# ── Export Q8_0 (higher quality, fits RTX 4080 16GB) ──
OUTPUT_DIR_Q8 = f"/content/{MODEL_NAME}-gguf-q8"
print("\nExporting Q8_0 quantization (higher quality)...")
try:
    model.save_pretrained_gguf(
        OUTPUT_DIR_Q8,
        tokenizer,
        quantization_method="q8_0",
    )
    has_q8 = True
except Exception as e:
    print(f"  Q8 export failed (may need more RAM): {e}")
    has_q8 = False

# Find all GGUF files (search recursively in case Unsloth nests them)
import glob
gguf_files = glob.glob(f"{OUTPUT_DIR}/**/*.gguf", recursive=True)
if not gguf_files:
    # Fallback: check for any large files that might be the model
    gguf_files = [f"{OUTPUT_DIR}/{fn}" for fn in os.listdir(OUTPUT_DIR)
                  if os.path.isfile(f"{OUTPUT_DIR}/{fn}") and os.path.getsize(f"{OUTPUT_DIR}/{fn}") > 1024**3]

gguf_files_q8 = []
if has_q8:
    gguf_files_q8 = glob.glob(f"{OUTPUT_DIR_Q8}/**/*.gguf", recursive=True)
    if not gguf_files_q8:
        gguf_files_q8 = [f"{OUTPUT_DIR_Q8}/{fn}" for fn in os.listdir(OUTPUT_DIR_Q8)
                         if os.path.isfile(f"{OUTPUT_DIR_Q8}/{fn}") and os.path.getsize(f"{OUTPUT_DIR_Q8}/{fn}") > 1024**3]

print(f"\nGGUF files found:")
for f in gguf_files + gguf_files_q8:
    size = os.path.getsize(f) / 1024**3
    print(f"  {os.path.basename(f)}: {size:.2f} GB")

if not gguf_files:
    print("\nWARNING: No GGUF files found! Check export output above for errors.")
    print("You can still use the LoRA adapter (saved in previous cell).")

In [ ]:
# ── Create Modelfiles for both quantizations ──
for label, gdir, gfiles in [("Q4_K_M", OUTPUT_DIR, gguf_files),
                              ("Q8_0", OUTPUT_DIR_Q8, gguf_files_q8)]:
    if not gfiles:
        continue
    gguf_name = os.path.basename(gfiles[0])

    modelfile_content = f"""FROM ./{gguf_name}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end|>"

SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"
"""
    mf_path = f"{gdir}/Modelfile"
    with open(mf_path, "w") as f:
        f.write(modelfile_content)
    print(f"{label} Modelfile created: {mf_path}")

print(f"\nAll files in {OUTPUT_DIR}:")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{fn}")
    if size > 1024**2:
        print(f"  {fn}: {size/1024**3:.2f} GB")
    else:
        print(f"  {fn}: {size/1024:.1f} KB")

## 10. Save & Download

**Option A:** Direct download (if GGUF < 4GB)  
**Option B:** Save to Google Drive (recommended for large files)

In [ ]:
# ── Also save the LoRA adapter (smaller, can merge locally) ──
adapter_dir = f"/content/{MODEL_NAME}-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"LoRA adapter saved to {adapter_dir}")

In [ ]:
# ── Option A: Direct download (if GGUF < 4GB) ──
from google.colab import files

if gguf_files:
    gguf_path = gguf_files[0]
    gguf_name = os.path.basename(gguf_path)
    gguf_size = os.path.getsize(gguf_path) / 1024**3
    if gguf_size < 4:
        print(f"Downloading {gguf_name} ({gguf_size:.1f} GB)...")
        files.download(gguf_path)
        modelfile_path = f"{OUTPUT_DIR}/Modelfile"
        if os.path.exists(modelfile_path):
            files.download(modelfile_path)
    else:
        print(f"GGUF is {gguf_size:.1f} GB — use Google Drive (Option B) instead.")
else:
    print("No GGUF file found. Use Option B (Google Drive) after checking the export cell.")

In [ ]:
# ── Option B: Save to Google Drive ──
from google.colab import drive
import shutil

drive.mount('/content/drive')

drive_dir = "/content/drive/MyDrive/runeclaw-model-v3"
os.makedirs(drive_dir, exist_ok=True)

# Copy Q4_K_M GGUF + Modelfile (skip directories like .cache)
for fn in os.listdir(OUTPUT_DIR):
    src = f"{OUTPUT_DIR}/{fn}"
    if os.path.isdir(src):
        print(f"  Skipping directory: {fn}")
        continue
    dst = f"{drive_dir}/{fn}"
    print(f"  Copying {fn}...")
    shutil.copy2(src, dst)

# Copy Q8_0 if available
if has_q8:
    drive_dir_q8 = f"{drive_dir}/q8"
    os.makedirs(drive_dir_q8, exist_ok=True)
    for fn in os.listdir(OUTPUT_DIR_Q8):
        src = f"{OUTPUT_DIR_Q8}/{fn}"
        if os.path.isdir(src):
            print(f"  Skipping directory: {fn}")
            continue
        dst = f"{drive_dir_q8}/{fn}"
        print(f"  Copying Q8/{fn}...")
        shutil.copy2(src, dst)

# Also copy training data for reference
shutil.copy2(DATA_FILE, f"{drive_dir}/training_data.jsonl")

print(f"\nSaved to Google Drive: {drive_dir}")
print(f"\n{'='*50}")
print(f"  DEPLOYMENT INSTRUCTIONS")
print(f"{'='*50}")
print(f"  1. Download folder from Google Drive")
print(f"  2. cd into the folder")
print(f"  3. ollama create runeclaw -f Modelfile")
print(f"  4. ollama run runeclaw 'Scan BTC/USDT for trade setups'")
print(f"\n  For RTX 4080 (higher quality Q8):")
print(f"  cd q8 && ollama create runeclaw-q8 -f Modelfile")
print(f"\n  For bot integration:")
print(f"  Set LLM_PROVIDER=ollama")
print(f"  Set LLM_MODEL=runeclaw")
print(f"  Set LLM_BASE_URL=http://localhost:11434/v1")